In [1]:
import pandas as pd
import numpy as np
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm

INPUT_FILE = "stratified_train.csv"
OUTPUT_FILE = "extracted_features_train.csv"
BINDER_THRESHOLD_NM = 500.0 

AA_GROUPS = {
    'charged': set('DEKHR'),
    'aliphatic': set('ILV'),
    'aromatic': set('FHWY'),
    'polar': set('DERKQN'),
    'neutral': set('AGHPSTY'),
    'hydrophobic': set('CVLIMFW'),
    'positively_charged': set('HKR'),
    'negatively_charged': set('DE'),
    'tiny': set('ACDGST'),
    'small': set('EHILKMNPQV'),
    'large': set('FRWY')
}

H_BOND_DONORS = {
    'A': 0, 'C': 0, 'D': 1, 'E': 1, 'F': 0, 'G': 0, 'H': 1, 
    'I': 0, 'K': 3, 'L': 0, 'M': 0, 'N': 2, 'P': 0, 'Q': 2, 
    'R': 5, 'S': 1, 'T': 1, 'V': 0, 'W': 1, 'Y': 1
}

def get_physiochemical_features(sequence, prefix):
    """
    Extracts comprehensive biochemical properties based on specific study parameters.
    """
    seq_str = str(sequence).upper().strip()
    clean_seq = seq_str.replace("X", "A").replace("U", "C").replace("Z", "E").replace("B", "D").replace("*", "")
    
    length = len(clean_seq)
    if length == 0:
        return {}

    features = {}
    
    try:
        analysed_seq = ProteinAnalysis(clean_seq)
        
        features[f'{prefix}length'] = length
        features[f'{prefix}mol_weight'] = analysed_seq.molecular_weight()
        features[f'{prefix}isoelectric_pt'] = analysed_seq.isoelectric_point() 
        features[f'{prefix}hydropathy_idx'] = analysed_seq.gravy() 
  
        features[f'{prefix}net_charge'] = analysed_seq.charge_at_pH(7.4)

        h_donors = sum(H_BOND_DONORS.get(aa, 0) for aa in clean_seq)
        features[f'{prefix}h_bond_donors'] = h_donors

        for group_name, amino_acids in AA_GROUPS.items():
            count = sum(1 for aa in clean_seq if aa in amino_acids)
            features[f'{prefix}comp_{group_name}'] = count / length


        aa_counts = analysed_seq.count_amino_acids()
        for aa in "ACDEFGHIKLMNPQRSTVWY":

            features[f'{prefix}pct_{aa}'] = aa_counts.get(aa, 0) / length
            
        return features
        
    except Exception as e:
        print(f"Warning: Error processing '{sequence}': {e}")
        return {f'{prefix}length': length}

print(f"--- Loading data from {INPUT_FILE} ---")
df = pd.read_csv(INPUT_FILE)

df = df.dropna(subset=['Epitope', 'HLA_seq', 'pIC50'])
print(f"Loaded {len(df)} rows.")

df['IC50_nM'] = 10**(9 - df['pIC50'])
df['target'] = (df['IC50_nM'] < BINDER_THRESHOLD_NM).astype(int)

print("\n--- Extracting Peptide Features... ---")

pep_data = [get_physiochemical_features(seq, "pep_") for seq in tqdm(df['Epitope'])]
pep_df = pd.DataFrame(pep_data)

print("--- Extracting HLA Features... ---")

hla_data = [get_physiochemical_features(seq, "hla_") for seq in tqdm(df['HLA_seq'])]
hla_df = pd.DataFrame(hla_data)

print("\n--- Combining features and saving... ---")
X = pd.concat([pep_df, hla_df], axis=1).fillna(0)
X['target'] = df['target'].values 

print(f"Feature Matrix Shape: {X.shape}")

X.to_csv(OUTPUT_FILE, index=False)
print(f"Done! Enhanced features saved to {OUTPUT_FILE}")

--- Loading data from stratified_train.csv ---
Loaded 327528 rows.

--- Extracting Peptide Features... ---


  0%|          | 0/327528 [00:00<?, ?it/s]

--- Extracting HLA Features... ---


  0%|          | 0/327528 [00:00<?, ?it/s]


--- Combining features and saving... ---
Feature Matrix Shape: (327528, 75)
Done! Enhanced features saved to extracted_features_train.csv


In [2]:
import pandas as pd
import numpy as np
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from tqdm.auto import tqdm

INPUT_FILE = "stratified_test.csv"
OUTPUT_FILE = "extracted_features_test.csv"
BINDER_THRESHOLD_NM = 500.0 

AA_GROUPS = {
    'charged': set('DEKHR'),
    'aliphatic': set('ILV'),
    'aromatic': set('FHWY'),
    'polar': set('DERKQN'),
    'neutral': set('AGHPSTY'),
    'hydrophobic': set('CVLIMFW'),
    'positively_charged': set('HKR'),
    'negatively_charged': set('DE'),
    'tiny': set('ACDGST'),
    'small': set('EHILKMNPQV'),
    'large': set('FRWY')
}

H_BOND_DONORS = {
    'A': 0, 'C': 0, 'D': 1, 'E': 1, 'F': 0, 'G': 0, 'H': 1, 
    'I': 0, 'K': 3, 'L': 0, 'M': 0, 'N': 2, 'P': 0, 'Q': 2, 
    'R': 5, 'S': 1, 'T': 1, 'V': 0, 'W': 1, 'Y': 1
}

def get_physiochemical_features(sequence, prefix):
    """
    Extracts comprehensive biochemical properties based on specific study parameters.
    """

    seq_str = str(sequence).upper().strip()

    clean_seq = seq_str.replace("X", "A").replace("U", "C").replace("Z", "E").replace("B", "D").replace("*", "")
    
    length = len(clean_seq)
    if length == 0:
        return {}

    features = {}
    
    try:
    
        analysed_seq = ProteinAnalysis(clean_seq)
        
        features[f'{prefix}length'] = length
        features[f'{prefix}mol_weight'] = analysed_seq.molecular_weight()
        features[f'{prefix}isoelectric_pt'] = analysed_seq.isoelectric_point() 
        features[f'{prefix}hydropathy_idx'] = analysed_seq.gravy() 
 
        features[f'{prefix}net_charge'] = analysed_seq.charge_at_pH(7.4)

        h_donors = sum(H_BOND_DONORS.get(aa, 0) for aa in clean_seq)
        features[f'{prefix}h_bond_donors'] = h_donors

        for group_name, amino_acids in AA_GROUPS.items():
            count = sum(1 for aa in clean_seq if aa in amino_acids)
            features[f'{prefix}comp_{group_name}'] = count / length

        aa_counts = analysed_seq.count_amino_acids()
        for aa in "ACDEFGHIKLMNPQRSTVWY":
      
            features[f'{prefix}pct_{aa}'] = aa_counts.get(aa, 0) / length
            
        return features
        
    except Exception as e:
        print(f"Warning: Error processing '{sequence}': {e}")
        return {f'{prefix}length': length}

print(f"--- Loading data from {INPUT_FILE} ---")
df = pd.read_csv(INPUT_FILE)

df = df.dropna(subset=['Epitope', 'HLA_seq', 'pIC50'])
print(f"Loaded {len(df)} rows.")

df['IC50_nM'] = 10**(9 - df['pIC50'])
df['target'] = (df['IC50_nM'] < BINDER_THRESHOLD_NM).astype(int)

print("\n--- Extracting Peptide Features... ---")

pep_data = [get_physiochemical_features(seq, "pep_") for seq in tqdm(df['Epitope'])]
pep_df = pd.DataFrame(pep_data)

print("--- Extracting HLA Features... ---")

hla_data = [get_physiochemical_features(seq, "hla_") for seq in tqdm(df['HLA_seq'])]
hla_df = pd.DataFrame(hla_data)

print("\n--- Combining features and saving... ---")
X = pd.concat([pep_df, hla_df], axis=1).fillna(0)
X['target'] = df['target'].values 

print(f"Feature Matrix Shape: {X.shape}")

X.to_csv(OUTPUT_FILE, index=False)
print(f"Done! Enhanced features saved to {OUTPUT_FILE}")

--- Loading data from stratified_test.csv ---
Loaded 18193 rows.

--- Extracting Peptide Features... ---


  0%|          | 0/18193 [00:00<?, ?it/s]

--- Extracting HLA Features... ---


  0%|          | 0/18193 [00:00<?, ?it/s]


--- Combining features and saving... ---
Feature Matrix Shape: (18193, 75)
Done! Enhanced features saved to extracted_features_test.csv
